# Charting Frequency and Difference

**Computational Text Analysis Pack · Module 10 — Visualising Text**

*Technique:* Bar charts, ranked comparisons, and the axis decisions that mislead

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/10-visualising-and-arguing/10a_charting_frequency.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Text visualisation has one job: show a comparison a reader can check. Most of the
decisions that decide whether it does are boring — orientation, sorting, what the
axis starts at, whether the numbers are normalised.

This notebook goes through them on real charts, and then builds the one figure
that is genuinely worth publishing: the keyness–effect-size scatter.

## The tooling

**matplotlib**, and the argument that the chart is a claim. **seaborn** and
**ggplot2** (R) are the usual higher-level layers; the decisions this notebook
is about — axis, baseline, ordering — are yours regardless of library.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
from collections import Counter

def frequencies(texts):
    """Token counts across a set of documents."""
    counts = Counter()
    for t in texts:
        counts.update(tokenise(t))
    return counts

def freq_frame(counts, top=25):
    total = sum(counts.values())
    rows = [{"rank": i, "word": w, "count": n, "per_1000": n / total * 1000}
            for i, (w, n) in enumerate(counts.most_common(top), start=1)]
    return pd.DataFrame(rows)

In [ ]:
LANGUAGE = "English"        # ← change this

sub = df[df["language"] == LANGUAGE]
counts = frequencies(sub["text"])
common = [w for w, _ in counts.most_common(60)][15:35]
vals = [counts[w] for w in common]

In [ ]:
# Three versions of one chart. Only the third is defensible.
fig, axes = plt.subplots(1, 3, figsize=(14, 4.6))

axes[0].bar(common, vals)
axes[0].set_title("vertical bars, unsorted")
axes[0].tick_params(axis="x", rotation=90, labelsize=7)

axes[1].bar(common, vals)
axes[1].set_ylim(min(vals) * 0.95, max(vals) * 1.02)
axes[1].set_title("truncated y-axis — differences look huge")
axes[1].tick_params(axis="x", rotation=90, labelsize=7)

order = np.argsort(vals)
axes[2].barh([common[i] for i in order], [vals[i] for i in order])
axes[2].set_title("horizontal, sorted, zero baseline")
axes[2].set_xlabel("occurrences")
plt.tight_layout(); plt.show()

Read the middle panel again. Nothing in it is false — the bars are the right
heights for the axis drawn. It is still the commonest way a chart lies, and it
happens by accident every time somebody zooms in to "show the differences more
clearly".

Word labels are long, so **horizontal bars, sorted, from zero.** That single
default fixes most text charts.

In [ ]:
def log_likelihood(counts_a, counts_b, min_count=10):
    """Words that distinguish corpus A from corpus B.

    The G-squared statistic corpus linguists call *keyness*. A word scores high
    when it is far more common in one corpus than its overall rate predicts.
    Positive `diff` means over-used in A.
    """
    na, nb = sum(counts_a.values()), sum(counts_b.values())
    rows = []
    for w in set(counts_a) | set(counts_b):
        o1, o2 = counts_a.get(w, 0), counts_b.get(w, 0)
        if o1 + o2 < min_count:
            continue
        e1 = na * (o1 + o2) / (na + nb)
        e2 = nb * (o1 + o2) / (na + nb)
        g2 = 2 * ((o1 * np.log(o1 / e1) if o1 else 0)
                  + (o2 * np.log(o2 / e2) if o2 else 0))
        rows.append({"word": w, "count_a": o1, "count_b": o2,
                     "per_1000_a": o1 / na * 1000, "per_1000_b": o2 / nb * 1000,
                     "keyness": g2, "over_used_in": "A" if o1 / na > o2 / nb else "B"})
    out = pd.DataFrame(rows).sort_values("keyness", ascending=False)
    return out.reset_index(drop=True)

In [ ]:
# The figure worth publishing: evidence against effect size.
TARGET = "health"           # ← change this

t = sub[sub["category"] == TARGET]
r = sub[sub["category"] != TARGET]
keys = log_likelihood(frequencies(t["text"]), frequencies(r["text"]), min_count=25)
keys["log_ratio"] = np.log2(keys["per_1000_a"].clip(lower=1e-4)
                            / keys["per_1000_b"].clip(lower=1e-4))

fig, ax = plt.subplots(figsize=(9.5, 6.5))
ax.scatter(keys["log_ratio"], keys["keyness"], s=16, alpha=0.3, color="grey")
notable = keys[(keys["keyness"] > keys["keyness"].quantile(0.98))
               & (keys["log_ratio"].abs() > 1)]
ax.scatter(notable["log_ratio"], notable["keyness"], s=26, color="indianred")
for _, row in notable.head(18).iterrows():
    ax.annotate(row["word"], (row["log_ratio"], row["keyness"]),
                fontsize=8, xytext=(4, 2), textcoords="offset points")
ax.axvline(0, color="grey", lw=0.8)
ax.set_xlabel(f"← other categories        log₂ rate ratio        {TARGET} →")
ax.set_ylabel("keyness (log-likelihood G²)")
ax.set_title(f"Words distinctive of {TARGET} reporting: evidence vs effect size")
plt.tight_layout(); plt.show()

## Why that chart works

- Both dimensions a reader needs are on the page, so nobody has to trust a ranking
- The zero line is drawn, so direction is unambiguous
- Only the labelled points are claims; the grey cloud is the context they sit in
- A sceptical reader can find a word they expected and see where it fell

That last property is what makes a figure evidence rather than decoration.

In [ ]:
# The paired-bar chart, for when you have a specific small set of words.
TERMS = ["hospital", "government", "vaccine", "school", "police", "market"]
na = sum(frequencies(t["text"]).values())
ca, cb = frequencies(t["text"]), frequencies(r["text"])
nb = sum(cb.values())

a_rates = [ca.get(w, 0) / na * 1000 for w in TERMS]
b_rates = [cb.get(w, 0) / nb * 1000 for w in TERMS]
y = np.arange(len(TERMS))

fig, ax = plt.subplots(figsize=(8.5, 4.5))
ax.barh(y - 0.2, a_rates, 0.4, label=TARGET)
ax.barh(y + 0.2, b_rates, 0.4, label="other categories")
ax.set_yticks(y); ax.set_yticklabels(TERMS)
ax.set_xlabel("occurrences per 1,000 words")
ax.set_title("Rates, not counts — the corpora are different sizes")
ax.legend()
plt.tight_layout(); plt.show()

### 🔧 Your turn 1

Change `TERMS` to words from your own research and re-run.

Then try it with raw counts instead of per-1,000 rates. The chart still draws, the
bars still have heights, and the comparison is meaningless — because the two
corpora are different sizes.

### 🔧 Your turn 2

Take the scatter plot and change which points get labelled: the top 5 by keyness,
then the top 5 by log ratio.

The figure tells a different story each time, from identical data. Labelling is an
argument. Say in your caption how you chose.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** With raw counts the larger corpus wins every comparison, and the
chart shows nothing but which category has more documents. This is the single
commonest error in student text-analysis work and it is invisible unless you check
the axis label — which is why the label should always read "per 1,000 words".

**Your turn 2.** Labelling by keyness surfaces common words with modest
differences; labelling by log ratio surfaces rare words with dramatic ones.
Neither is wrong and neither is neutral. A caption saying "the 18 words with
G² above the 98th percentile and |log₂ ratio| > 1 are labelled" lets a reader
reconstruct your choice, which is all that is being asked of you.

</details>

## Where this stops

You can make an honest frequency figure. Next: **10b**, on the chart that is
almost always the wrong one.